# 09 - SFT redo with the training bugs fixed

## 1. Installations

In [ ]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.11.2', 'triton') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
try:
    import importlib.metadata as _md; _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
except Exception:
    _torch_v = ()
_torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
!uv pip install -qqq --no-deps --upgrade "{_torchao}"
!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip install -qqq math-verify
print("is_t4:", is_t4, "| vllm:", _vllm)

## 2. Environment check

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_TF"] = "1"; os.environ["USE_TF"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import importlib.metadata as md
for pkg in ["torch", "vllm", "transformers", "trl", "peft", "unsloth", "datasets", "math-verify"]:
    try: print(f"{pkg:13s} {md.version(pkg)}")
    except md.PackageNotFoundError: print(f"{pkg:13s} NOT INSTALLED")

import torch
print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 3. Shared extractor + eval helpers

In [ ]:
%%writefile rf_extract.py
import re

BOXED_SUFFIX = "\nPlease reason step by step, and put your final answer within \\boxed{}."


def user_prompt(question, style="boxed"):
    if style == "boxed":
        return question + BOXED_SUFFIX
    if style == "plain":
        return question
    raise ValueError(style)


def _boxed_spans(text):
    key = r"\boxed{"
    spans, i = [], 0
    while True:
        j = text.find(key, i)
        if j == -1:
            break
        k = j + len(key)
        start, depth = k, 1
        while k < len(text) and depth > 0:
            if text[k] == "{":
                depth += 1
            elif text[k] == "}":
                depth -= 1
            k += 1
        spans.append(text[start:k - 1])
        i = k
    return spans


def _unwrap_text(s):
    return re.sub(r"\\(?:text|textbf|mbox|mathrm)\{([^{}]*)\}", r"\1", s)


def norm(x):
    if x is None:
        return None
    s = str(x).strip()
    s = _unwrap_text(s)
    s = s.replace("\\!", "").replace("\\,", "").replace("\\;", "").replace(" ", "")
    s = s.replace("\\$", "").replace("$", "")
    s = re.sub(r"(?<=\d),(?=\d{3}\b)", "", s)
    s = s.replace("\\%", "").replace("%", "")
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace("^\\circ", "").replace("^{\\circ}", "")
    s = s.rstrip(".").strip()
    if s.startswith("{") and s.endswith("}"):
        s = s[1:-1]
    return s


def extract_boxed(text):
    boxes = _boxed_spans(text)
    return norm(boxes[-1]) if boxes else None


def extract_answer(text):
    boxed = extract_boxed(text)
    if boxed is not None:
        return boxed
    nums = re.findall(r"-?\d+\.?\d*", text.replace(",", ""))
    return norm(nums[-1]) if nums else None


def _to_float(s):
    try:
        return float(s)
    except (TypeError, ValueError):
        return None


def _math_verify(pred, gold):
    try:
        from math_verify import parse, verify
    except ImportError:
        return False
    try:
        return bool(verify(parse("\\boxed{" + gold + "}"), parse("\\boxed{" + pred + "}")))
    except Exception:
        return False


def answers_match(pred, gold):
    if pred is None or gold is None:
        return False
    p, g = norm(pred), norm(gold)
    if not p or not g:
        return False
    if p == g:
        return True
    gf = _to_float(g)
    if gf is not None:
        m = re.fullmatch(r"(-?\d+(?:\.\d+)?)[a-zA-Z]*", p)
        pf = _to_float(m.group(1)) if m else None
        return pf is not None and abs(pf - gf) < 1e-6
    return _math_verify(p, g)


def gsm8k_gold(answer_field):
    return norm(answer_field.split("####")[-1])


def has_boxed(text):
    return bool(_boxed_spans(text))

In [ ]:
%%writefile rf_eval.py
import os
import pickle
import random
from collections import Counter

from rf_extract import answers_match, extract_answer, extract_boxed, gsm8k_gold, has_boxed, user_prompt

DEV_SIZE = 500


def gsm8k_train(split=None, seed=0):
    from datasets import load_dataset
    ds = load_dataset("openai/gsm8k", "main", split="train")
    order = list(range(len(ds)))
    random.Random(seed).shuffle(order)
    dev = set(order[:DEV_SIZE])
    items = [{"id": i, "split": "dev" if i in dev else "train",
              "question": r["question"], "gold": gsm8k_gold(r["answer"])}
             for i, r in enumerate(ds)]
    return [it for it in items if split in (None, it["split"])]


def load_eval_set(name, n=None):
    from datasets import load_dataset
    if name == "gsm8k":
        ds = load_dataset("openai/gsm8k", "main", split="test")
        items = [{"question": r["question"], "gold": gsm8k_gold(r["answer"])} for r in ds]
    elif name == "gsm8k_dev":
        items = gsm8k_train("dev")
    elif name == "math500":
        ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
        items = [{"question": r["problem"], "gold": r["answer"],
                  "subject": r["subject"], "level": r["level"]} for r in ds]
    else:
        raise ValueError(name)
    return items[:n] if n else items


def chat_prompts(tokenizer, items, style):
    return [tokenizer.apply_chat_template([{"role": "user", "content": user_prompt(it["question"], style)}],
                                          tokenize=False, add_generation_prompt=True)
            for it in items]


def is_looping(text, tail=3000, n=8, min_repeats=5):
    words = text[-tail:].split()
    if len(words) < 50:
        return False
    grams = Counter(" ".join(words[i:i + n]) for i in range(len(words) - n))
    return grams.most_common(1)[0][1] >= min_repeats


def completion_row(c):
    return {"resp": c.text, "n_new_tokens": len(c.token_ids),
            "truncated": int(c.finish_reason == "length"),
            "has_box": has_boxed(c.text), "looping": is_looping(c.text)}


def rows_from_outputs(items, outputs):
    return [{**it, **completion_row(out.outputs[0])} for it, out in zip(items, outputs)]


def graded(c, gold):
    row = completion_row(c)
    row["pred"] = extract_boxed(c.text)
    row["reward"] = int(answers_match(row["pred"], gold))
    return row


def sample_rows(items, greedy_outputs, sampled_outputs):
    return [{**it, "greedy": graded(g.outputs[0], it["gold"]),
             "samples": [graded(c, it["gold"]) for c in s.outputs]}
            for it, g, s in zip(items, greedy_outputs, sampled_outputs)]


def majority_correct(row):
    preds = [s["pred"] for s in row["samples"] if s["pred"] is not None]
    return bool(preds) and answers_match(Counter(preds).most_common(1)[0][0], row["gold"])


def sample_summary(rows):
    n, k = len(rows), len(rows[0]["samples"])
    passes = [sum(s["reward"] for s in r["samples"]) for r in rows]
    samples = [s for r in rows for s in r["samples"]]
    tokens = sorted(s["n_new_tokens"] for s in samples)
    return {
        "n": n, "k": k,
        "greedy": sum(r["greedy"]["reward"] for r in rows) / n,
        "sample_mean": sum(passes) / (n * k),
        "majority": sum(majority_correct(r) for r in rows) / n,
        "pass_any": sum(p > 0 for p in passes) / n,
        "mixed": sum(0 < p < k for p in passes) / n,
        "hist": [passes.count(i) for i in range(k + 1)],
        "truncated": sum(s["truncated"] for s in samples) / len(samples),
        "looping": sum(s["looping"] for s in samples) / len(samples),
        "median_tokens": tokens[len(tokens) // 2],
    }


def quick_acc(rows):
    return sum(answers_match(extract_answer(r["resp"]), r["gold"]) for r in rows) / max(len(rows), 1)


def load_result(path, config):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return {"config": config}


def save_result(path, result):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(result, f)
    os.replace(tmp, path)

## 4. Config

In [ ]:
MODE = "smoke"
DATA = "self"

MODEL_ID          = "unsloth/Qwen2.5-1.5B-Instruct"
LORA_R, LORA_ALPHA = 32, 64
LR                = 2e-4
BATCH             = 1
GRAD_ACCUM        = 16
MAX_LEN           = {"self": 1536, "r1": 4096}[DATA]
EVAL_MAX_NEW      = 2048

N_EXAMPLES, SAVE_STEPS, EVAL_N, TIME_BUDGET_HOURS = {
    "smoke": (80, 5, 50, 1.0),
    "full":  ({"self": None, "r1": 10000}[DATA], 100, None, 9.0),
}[MODE]

RUN_NAME  = f"sft2_{DATA}_{MODE}"
OUT_DIR   = f"/kaggle/working/{RUN_NAME}"
FINAL_DIR = f"/kaggle/working/{RUN_NAME}-final"
RES_DIR   = "/kaggle/working/results"
os.makedirs(RES_DIR, exist_ok=True)
print(f"{MODE} on {DATA}: examples={N_EXAMPLES or 'all'}  examples/step={BATCH*GRAD_ACCUM}  max_len={MAX_LEN}")

## 5. Load model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_LEN,
    load_in_4bit=False,
    fast_inference=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("eos:", tokenizer.eos_token, "| pad:", tokenizer.pad_token)
print(f"allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 6. Training data

In [ ]:
import glob, pickle, random
from datasets import Dataset, load_from_disk
from rf_extract import user_prompt

rng = random.Random(42)
samples_path = glob.glob("/kaggle/input/**/gsm8k_train_k8.pkl", recursive=True)[0]
rows = pickle.load(open(samples_path, "rb"))["rows"]
dev_rows = [r for r in rows if r["split"] == "dev"]

if DATA == "self":
    examples = []
    for r in rows:
        if r["split"] != "train":
            continue
        right = [s for s in r["samples"] if s["reward"] and not s["truncated"]]
        if right:
            prompt = tokenizer.apply_chat_template([{"role": "user", "content": user_prompt(r["question"], "boxed")}],
                                                   tokenize=False, add_generation_prompt=True)
            examples.append({"prompt": prompt, "completion": rng.choice(right)["resp"]})
else:
    r1_dir = os.path.dirname(glob.glob("/kaggle/input/**/train_ds/dataset_info.json", recursive=True)[0])
    marker = "<|im_start|>assistant\n"
    examples = []
    for text in load_from_disk(r1_dir)["text"]:
        head, tail = text.split(marker, 1)
        examples.append({"prompt": head + marker, "completion": tail.rstrip()})

rng.shuffle(examples)
examples = examples[:N_EXAMPLES] if N_EXAMPLES else examples
train = Dataset.from_list(examples)

lens = sorted(len(tokenizer(e["completion"]).input_ids) for e in examples[:500])
print(f"{DATA}: {len(train)} examples; completion median {lens[len(lens)//2]} tokens, p95 {lens[int(len(lens)*0.95)]}")
print(train[0]["prompt"][-120:], "...")
print(train[0]["completion"][:200], "...")

## 7. Trainer

In [ ]:
import json, re, time
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

def step_of(path):
    return int(re.search(r"checkpoint-(\d+)", path).group(1))

def latest_checkpoint():
    cands = glob.glob("/kaggle/input/**/checkpoint-*/trainer_state.json", recursive=True)
    cands += glob.glob(f"{OUT_DIR}/checkpoint-*/trainer_state.json")
    cands = [os.path.dirname(c) for c in cands if RUN_NAME in c]
    return max(cands, key=step_of, default=None)

RESUME = None if MODE == "smoke" else latest_checkpoint()
print("resume from:", RESUME)

class TimeBudgetAndLogs(TrainerCallback):
    def __init__(self, hours):
        self.deadline = time.time() + hours * 3600
        self.t_last = None
    def on_step_end(self, args, state, control, **kw):
        now = time.time()
        if self.t_last is not None and state.global_step % 10 == 0:
            print(f"step {state.global_step}: {now - self.t_last:.0f}s/step")
        self.t_last = now
        if now > self.deadline:
            print(f"time budget reached at step {state.global_step} -> saving and stopping")
            control.should_save = True
            control.should_training_stop = True
    def on_save(self, args, state, control, **kw):
        with open(f"{args.output_dir}/log_history.json", "w") as f:
            json.dump(state.log_history, f)

args = SFTConfig(
    output_dir=OUT_DIR,
    run_name=RUN_NAME,
    report_to="none",
    seed=42,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_8bit",
    weight_decay=0.0,
    max_grad_norm=1.0,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=1,
    max_length=MAX_LEN,
    completion_only_loss=True,
    packing=False,
    save_steps=SAVE_STEPS,
    save_total_limit=None,
    logging_steps=1,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train,
    processing_class=tokenizer,
    callbacks=[TimeBudgetAndLogs(TIME_BUDGET_HOURS)],
)

batch = next(iter(trainer.get_train_dataloader()))
labels = batch["labels"][0]
print("tokens in loss:", int((labels != -100).sum()), "of", labels.numel())
print("last label token:", tokenizer.decode([int(labels[labels != -100][-1])]))

## 8. Train

In [ ]:
t0 = time.time()
trainer.train(resume_from_checkpoint=RESUME)
print(f"training wall time: {(time.time()-t0)/3600:.2f} h")

model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
with open(f"{FINAL_DIR}/log_history.json", "w") as f:
    json.dump(trainer.state.log_history, f)
log_history = trainer.state.log_history

## 9. Training curve

In [ ]:
hist = [h for h in log_history if "loss" in h]
every = max(1, len(hist) // 20)
print(f"{'step':>5s} {'loss':>8s} {'grad_norm':>10s} {'lr':>10s}")
for h in hist[::every] + ([hist[-1]] if hist[-1] not in hist[::every] else []):
    print(f"{h['step']:5d} {h['loss']:8.3f} {h.get('grad_norm', float('nan')):10.3f} {h.get('learning_rate', 0):10.2e}")

## 10. Dev accuracy per checkpoint

In [ ]:
import gc, shutil
from rf_eval import chat_prompts, graded, save_result

del trainer, model
gc.collect()
torch.cuda.empty_cache()
print(f"allocated after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model=MODEL_ID,
    dtype="half",
    max_model_len=EVAL_MAX_NEW + 512,
    gpu_memory_utilization=0.6,
    enable_lora=True,
    max_lora_rank=LORA_R,
    max_loras=1,
    seed=0,
)
sp = SamplingParams(temperature=0.0, max_tokens=EVAL_MAX_NEW, repetition_penalty=1.0)
dev_items = dev_rows[:EVAL_N] if EVAL_N else dev_rows
prompts = chat_prompts(tokenizer, dev_items, "boxed")

ckpts = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"), key=step_of)
runs = [("base", None)] + [(f"step{step_of(c)}", c) for c in ckpts] + [("final", FINAL_DIR)]

dev = {"config": {"n": len(dev_items), "max_new": EVAL_MAX_NEW, "mode": MODE, "data": DATA}}
print(f"{'model':>10s} {'dev acc':>8s} {'trunc':>6s} {'loop':>6s} {'tokens':>7s}")
for i, (name, path) in enumerate(runs):
    lora = LoRARequest(name, i + 1, path) if path else None
    outs = llm.generate(prompts, sp, lora_request=lora)
    dev[name] = [{**it, **graded(o.outputs[0], it["gold"])} for it, o in zip(dev_items, outs)]
    rows_ = dev[name]
    acc = sum(r["reward"] for r in rows_) / len(rows_)
    trunc = sum(r["truncated"] for r in rows_) / len(rows_)
    loop = sum(r["looping"] for r in rows_) / len(rows_)
    toks = sorted(r["n_new_tokens"] for r in rows_)[len(rows_) // 2]
    print(f"{name:>10s} {acc*100:7.1f}% {trunc*100:5.1f}% {loop*100:5.1f}% {toks:7d}")
    save_result(f"{RES_DIR}/dev_{RUN_NAME}.pkl", dev)

## 11. Package for download

In [ ]:
shutil.make_archive(f"/kaggle/working/{RUN_NAME}_adapter", "zip", FINAL_DIR)
shutil.make_archive("/kaggle/working/results", "zip", "/kaggle/working", "results")
print(f"download: {RUN_NAME}_adapter.zip, results.zip")